# Car Price Prediction

Predict a used vehicle's selling price in lakhs.

## 1. Import libraries

In [1]:
import datetime
import joblib
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

C:\ProgramData\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## 2. Load and inspect the dataset

In [2]:
df = pd.read_csv("car data.csv")

print("Dataset shape:", df.shape)
display(df.head())
display(df.isnull().sum())

Dataset shape: (301, 9)


,Car_Name,Year,Selling_Price,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner
0,ritz,2014,3.35,5.59,27000,Petrol,Dealer,Manual,0
1,sx4,2013,4.75,9.54,43000,Diesel,Dealer,Manual,0
2,ciaz,2017,7.25,9.85,6900,Petrol,Dealer,Manual,0
3,wagon r,2011,2.85,4.15,5200,Petrol,Dealer,Manual,0
4,swift,2014,4.60,6.87,42450,Diesel,Dealer,Manual,0


Car_Name         0
Year             0
Selling_Price    0
Present_Price    0
Kms_Driven       0
Fuel_Type        0
Seller_Type      0
Transmission     0
Owner            0
dtype: int64

## 3. Prepare features and target

We remove `Car_Name`, because the model in this project does not use it. We calculate `Age` from the manufacture year, then drop the original `Year` column.

In [3]:
data = df.drop(columns=["Car_Name"]).copy()

current_year = datetime.datetime.now().year
data["Age"] = current_year - data["Year"]
data = data.drop(columns=["Year"])

x = data.drop(columns=["Selling_Price"])
y = data["Selling_Price"]

display(x.head())
print("Input columns:", x.columns.tolist())

,Present_Price,Kms_Driven,Fuel_Type,Seller_Type,Transmission,Owner,Age
0,5.59,27000,Petrol,Dealer,Manual,0,12
1,9.54,43000,Diesel,Dealer,Manual,0,13
2,9.85,6900,Petrol,Dealer,Manual,0,9
3,4.15,5200,Petrol,Dealer,Manual,0,15
4,6.87,42450,Diesel,Dealer,Manual,0,12


Input columns: ['Present_Price', 'Kms_Driven', 'Fuel_Type', 'Seller_Type', 'Transmission', 'Owner', 'Age']


## 4. Define numeric and categorical columns

In [4]:
numeric_features = ["Age","Present_Price","Kms_Driven","Owner"]

categorical_features = ["Fuel_Type","Seller_Type","Transmission"]

preprocessor = ColumnTransformer(transformers=[
        ("numeric", "passthrough", numeric_features),
        ("categorical",OneHotEncoder(handle_unknown="ignore"),categorical_features)],
    remainder="drop")

## 5. Create the pipeline

The pipeline first preprocesses the columns, then trains the Random Forest model. During prediction, the same saved preprocessing is automatically applied.

In [5]:
model_pipeline = Pipeline([("preprocessor", preprocessor),
                            ("model", RandomForestRegressor(random_state=0))])

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=0)

model_pipeline.fit(x_train, y_train)

print("Pipeline training complete.")

Pipeline training complete.


## 6. Evaluate the model

In [6]:
train_predictions = model_pipeline.predict(x_train)
test_predictions = model_pipeline.predict(x_test)

print("Train R2:", r2_score(y_train, train_predictions))
print("Test R2:", (r2_score(y_test, test_predictions)))
print("Test MAE:",(mean_absolute_error(y_test, test_predictions)))
print("Test RMSE:",(mean_squared_error(y_test, test_predictions) ** 0.5))

Train R2: 0.9866585360134683
Test R2: 0.9582270340853795
Test MAE: 0.6117426229508196
Test RMSE: 1.0275741175453863


## 7. Save the complete pipeline

This saves both the fitted `ColumnTransformer` and the Random Forest model in one file.

In [7]:
joblib.dump(model_pipeline, "model.pkl")
print("Saved complete pipeline to model.pkl")

Saved complete pipeline to model.pkl


## 8. Load and test the saved pipeline

In [8]:
loaded_model = joblib.load("model.pkl")

sample_car = pd.DataFrame([{
    "Age": datetime.datetime.now().year - 2018,
    "Present_Price": 8.93,
    "Kms_Driven": 83000,
    "Owner": 0,
    "Fuel_Type": "Petrol",
    "Seller_Type": "Individual",
    "Transmission": "Manual"
}])

prediction = loaded_model.predict(sample_car)[0]
print(f"Predicted selling price: {prediction:.2f} lakhs")

Predicted selling price: 7.19 lakhs


## 9. Run the Flask app

Make sure these files are in the same project folder:

- `Car_Price_Prediction_Pipeline.ipynb`
- `car data.csv`
- `main.py`
- `model.pkl` (created when you run the training notebook)
- `templates/index.html` (your existing HTML form)

Install dependencies if needed:

```bash
pip install pandas scikit-learn flask joblib jupyter
```

Run `main.py` from that folder:

```bash
python main.py
```

The HTML form must use these exact field names: `Year`, `Present_Price`, `Kms_Driven`, `Owner`, `Fuel_Type`, `Seller_Type`, and `Transmission`.